# Phase 2: Data Enrichment via Web Scraping

| | |
|---|---|
| **What** | Change the ICD-9 codes in `diag_1` (like `428`) into disease names by scraping a website. |
| **Why** | A code like `428` means nothing to a business reader. "Heart failure" is easy to understand. |

## Part 1: Loading the Data

| | |
|---|---|
| **What** | Import the libraries and load `01_cleaned.csv` from Phase 1. `diag_1` is read as text. |
| **Why** | Some codes have letters or decimals (`V57`, `250.8`). As numbers, they could get changed. |

In [13]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup

# Load Phase 1 output, keep diag_1 as text and 'None' labels as text
df = pd.read_csv("../data/processed/01_cleaned.csv", dtype={"diag_1": str},
                 keep_default_na=False, na_values=[''], low_memory=False)

df.shape

(100114, 49)

## Part 2: Fixing the Leading Zeros

| | |
|---|---|
| **What** | Pad number codes to 3 digits (`38` → `038`, `8` → `008`). `V`, `E` and decimal codes stay the same. |
| **Why** | ICD-9 number codes have 3 digits, but the zeros were lost earlier. Without them the website cannot find codes like `038` (Septicemia). |
| **Checked** | The data has no short decimal codes like `8.45`, so this fix is safe. |

In [14]:
# Pad numeric codes to 3 digits: '38' -> '038' , '8' -> '008'
# V/E codes (V57) and decimal codes (250.8) stay the same

df["diag_1"] = df["diag_1"].apply(
    lambda c: c.zfill(3) if isinstance(c, str) and c[:1].isdigit() else c
)

df["diag_1"].head()

0    250.83
1       276
2       648
3       008
4       197
Name: diag_1, dtype: str

## Part 3: Top 20 Codes

| | |
|---|---|
| **What** | Count each `diag_1` code and pick the 20 most common ones as scraping targets. |
| **Why** | There are hundreds of codes. Scraping all of them would be slow, and the top 20 already cover 50.8% of the rows. |

In [15]:
# Frequncy of every diag_1 code
code_counts = df["diag_1"].value_counts()

# Top 20 most frequent codes are the scraping targets
top_20 = code_counts.head(20).index.tolist()
print(code_counts.head(20))

# Share of all rows covered by the top 20 codes
coverage = df["diag_1"].isin(top_20).mean()
print(f"\nTop 20 codes cover {coverage:.1%} of all rows")

diag_1
428      6735
414      6555
786      4016
410      3477
486      3413
427      2729
491      2252
715      2147
682      2030
780      2012
434      1958
996      1944
276      1861
250.8    1667
599      1581
038      1522
584      1482
V57      1204
250.6    1178
820      1075
Name: count, dtype: int64

Top 20 codes cover 50.8% of all rows


## Part 4: The Scraper

| | |
|---|---|
| **What** | A function `scrape_icd9_description(code)` searches one code on http://icd9.chrisendres.com/. It uses `requests` to get the page and `BeautifulSoup` to read it. A loop runs it for all 20 codes. |
| **Why** | The brief allows this site. It needs only 1 search per code, and the code and its name are on the same line, so a simple regex works. |
| **Key points** | `time.sleep(1)` between requests, so the site is not overloaded.<br>`try/except` and a timeout, so one failed request does not stop the loop.<br>so the website knows the requests come from a student project. |

In [16]:
headers = {"User-Agent": "Mozilla/5.0 (student project)"}


def scrape_icd9_description(code):
    """Return the disease name for one ICD-9 code, or None if not found."""
    url = (f"http://icd9.chrisendres.com/index.php?srchtype=diseases"
           f"&srchtext={code}&Submit=Search&action=search")
    try:
        # Download the search result page
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()

        # Find the line starting with the exact code and keep the text after it
        text = BeautifulSoup(response.text, "html.parser").get_text()
        for line in text.split("\n"):
            match = re.match(rf"\s*{re.escape(code)}\s+(\S.*)", line)
            if match:
                return match.group(1).strip()
    except requests.RequestException as e:
        print(f"Error scraping {code}: {e}")
    return None


scraped_data = {}
for code in top_20:
    scraped_data[code] = scrape_icd9_description(code)
    print(f"{code} -> {scraped_data[code]}")

    # Wait 1 second between requests
    time.sleep(1)

428 -> Heart failure
414 -> Other forms of chronic ischemic heart disease
786 -> Symptoms involving respiratory system and other chest symptoms
410 -> Acute myocardial infarction
486 -> Pneumonia, organism unspecified
427 -> Cardiac dysrhythmias
491 -> Chronic bronchitis
715 -> Osteoarthrosis and allied disorders
682 -> Other cellulitis and abscess
780 -> General symptoms
434 -> Occlusion of cerebral arteries
996 -> Complications peculiar to certain specified procedures
276 -> Disorders of fluid, electrolyte, and acid-base balance
250.8 -> Diabetes with other specified manifestations
599 -> Other disorders of urethra and urinary tract
038 -> Septicemia
584 -> Acute renal failure
V57 -> Care involving use of rehabilitation procedures
250.6 -> Diabetes with neurological manifestations
820 -> Fracture of neck of femur
